In [1]:
from PIL import Image, ImageOps
import qrcode
from qrcode.constants import ERROR_CORRECT_H

def generate_qr_with_logo(data: str,
                          logo_path: str,
                          output_path: str = "qr_with_logo.png",
                          qr_size: int = 800,
                          logo_scale: float = 0.20,
                          qr_fill_color: str = "#0A74DA",
                          qr_back_color: str = "white",
                          round_logo: bool = True):
    """
    Generate a colored QR code with an image (logo) embedded in center.

    Parameters:
    - data: the URL or text to encode
    - logo_path: path to logo image (PNG/JPG). Transparent PNG works best.
    - output_path: where to save final PNG
    - qr_size: final QR image pixel size (square)
    - logo_scale: fraction of QR width the logo should occupy (0.15-0.30 recommended)
    - qr_fill_color: color of QR modules (can be hex or color name)
    - qr_back_color: background color
    - round_logo: if True, make the logo circular (aesthetics)
    """

    # 1) Create QR object with high error correction
    qr = qrcode.QRCode(
        error_correction=ERROR_CORRECT_H,
        box_size=10,
        border=4
    )
    qr.add_data(data)
    qr.make(fit=True)

    # 2) Make base colored QR as PIL image
    qr_img = qr.make_image(fill_color=qr_fill_color, back_color=qr_back_color).convert("RGBA")

    # Resize to requested qr_size
    qr_img = qr_img.resize((qr_size, qr_size), Image.LANCZOS)

    # 3) Load logo
    logo = Image.open(logo_path).convert("RGBA")

    # Optionally make logo circular
    if round_logo:
        # ensure square crop
        min_side = min(logo.size)
        logo = ImageOps.fit(logo, (min_side, min_side), centering=(0.5, 0.5))
        # create circular mask
        mask = Image.new("L", logo.size, 0)
        draw = Image.new("L", logo.size, 0)
        from PIL import ImageDraw
        mask_draw = ImageDraw.Draw(mask)
        mask_draw.ellipse((0, 0, logo.size[0], logo.size[1]), fill=255)
        logo.putalpha(mask)

    # 4) Resize logo to be a fraction of QR size
    logo_max_width = int(qr_size * logo_scale)
    # keep aspect ratio
    logo_w, logo_h = logo.size
    logo_ratio = logo_h / logo_w
    new_logo_w = logo_max_width
    new_logo_h = int(new_logo_w * logo_ratio)
    logo = logo.resize((new_logo_w, new_logo_h), Image.LANCZOS)

    # 5) Paste logo onto center of QR; use logo alpha channel as mask
    pos = ((qr_img.size[0] - logo.size[0]) // 2, (qr_img.size[1] - logo.size[1]) // 2)
    qr_img.paste(logo, pos, mask=logo)

    # 6) Save
    qr_img.save(output_path)
    print(f"Saved QR with logo to: {output_path}")

if __name__ == "__main__":
    # --- user inputs ---
    data = "https://sakthijai0005-lang.github.io/revathi/"    # <-- replace with your link
    logo_path = "SM247500.JPG"                       # <-- replace with your image path
    output = "my_colored_qr.png"

    # generate
    generate_qr_with_logo(
        data=data,
        logo_path=logo_path,
        output_path=output,
        qr_size=1000,           # bigger for higher quality
        logo_scale=0.22,        # logo ~22% of QR width
        qr_fill_color="#1a9a7a",# change to any color you'd like
        qr_back_color="white",
        round_logo=True
    )


Saved QR with logo to: my_colored_qr.png


In [2]:
import segno
from PIL import Image

def generate_photo_qr(data, photo_path, output_path="photo_qr.png", qr_size=800, scale=10):
    """
    Generate a QR code blended with a photo background.

    Parameters:
    - data: The link or text to encode
    - photo_path: Path to your photo (PNG/JPG)
    - output_path: File to save QR code
    - qr_size: final QR image size
    - scale: controls how bold the QR is drawn
    """
    # 1. Generate QR in transparent mode
    qr = segno.make(data, error='h')  # high error correction
    qr.save("temp_qr.png", scale=scale, border=1, light=None, dark="black")

    # 2. Open images
    qr_img = Image.open("temp_qr.png").convert("RGBA")
    photo = Image.open(photo_path).convert("RGBA")

    # 3. Resize both to same size
    photo = photo.resize((qr_size, qr_size), Image.LANCZOS)
    qr_img = qr_img.resize((qr_size, qr_size), Image.LANCZOS)

    # 4. Blend photo with QR (QR will mask parts of photo)
    combined = Image.alpha_composite(photo, qr_img)

    # 5. Save
    combined.save(output_path)
    print(f"✅ Photo QR saved to {output_path}")

if __name__ == "__main__":
    generate_photo_qr(
        data="https://sakthijai0005-lang.github.io/revathi/",   # 🔗 replace with your link
        photo_path="SM247500.JPG",            # 📸 replace with your image file
        output_path="photo_qr.png",
        qr_size=1000
    )


✅ Photo QR saved to photo_qr.png


In [4]:
import qrcode
from PIL import Image

def generate_full_photo_qr(data, photo_path, output_path="full_photo_qr.png",
                           qr_size=800, transparency=180):
    """
    Generate a QR code overlaying your full photo as the background.

    Parameters:
    - data: The URL/text to encode
    - photo_path: Your background photo (PNG/JPG)
    - output_path: Where to save the QR code
    - qr_size: Output image size (pixels, square)
    - transparency: Darkness of QR modules (0=transparent, 255=solid black)
    """
    # 1. Create QR with transparent background
    qr = qrcode.QRCode(
        version=1,
        error_correction=qrcode.constants.ERROR_CORRECT_H,
        box_size=10,
        border=4,
    )
    qr.add_data(data)
    qr.make(fit=True)

    # Make QR with black fill and transparent background
    qr_img = qr.make_image(fill_color="black", back_color=None).convert("RGBA")
    qr_img = qr_img.resize((qr_size, qr_size), Image.LANCZOS)

    # Reduce opacity so background photo shows through
    datas = qr_img.getdata()
    new_data = []
    for item in datas:
        if item[3] > 0:  # non-transparent pixel
            new_data.append((0, 0, 0, transparency))  # semi-transparent black
        else:
            new_data.append((255, 255, 255, 0))  # fully transparent
    qr_img.putdata(new_data)

    # 2. Open and resize photo to match QR
    photo = Image.open(photo_path).convert("RGBA")
    photo = photo.resize((qr_size, qr_size), Image.LANCZOS)

    # 3. Overlay QR on photo
    combined = Image.alpha_composite(photo, qr_img)

    # 4. Save
    combined.save(output_path)
    print(f"✅ Full photo QR saved at {output_path}")

if __name__ == "__main__":
    generate_full_photo_qr(
        data="https://sakthijai0005-lang.github.io/revathi/",  # 🔗 replace with your link
        photo_path="SM247500.JPG",           # 📸 replace with your image
        output_path="full_photo_qr.png",
        qr_size=1000,
        transparency=500                       # adjust darkness (150–220 works best)
    )


✅ Full photo QR saved at full_photo_qr.png


In [5]:
import qrcode
from PIL import Image

def generate_photo_pattern_qr(data, photo_path, output_path="photo_pattern_qr.png", qr_size=800):
    """
    Generate a QR code where the photo fills the QR modules (not just background).
    """
    # Create QR in black & white (no background)
    qr = qrcode.QRCode(
        version=None,
        error_correction=qrcode.constants.ERROR_CORRECT_H,
        box_size=10,
        border=1,
    )
    qr.add_data(data)
    qr.make(fit=True)

    # Get QR matrix (list of lists of True/False)
    matrix = qr.get_matrix()
    n_rows, n_cols = len(matrix), len(matrix[0])

    # Resize photo to match QR grid
    photo = Image.open(photo_path).convert("RGB")
    photo = photo.resize((n_cols, n_rows), Image.LANCZOS)

    # Create output image
    out_img = Image.new("RGB", (n_cols, n_rows), "white")

    # Fill only QR modules with photo pixels
    for y in range(n_rows):
        for x in range(n_cols):
            if matrix[y][x]:  # if QR module is dark
                out_img.putpixel((x, y), photo.getpixel((x, y)))

    # Scale up for better visibility
    out_img = out_img.resize((qr_size, qr_size), Image.NEAREST)
    out_img.save(output_path)
    print(f"✅ Photo-pattern QR saved at {output_path}")

if __name__ == "__main__":
    generate_photo_pattern_qr(
        data="https://sakthijai0005-lang.github.io/revathi/",  # 🔗 replace with your link
        photo_path="SM247500.JPG",         # your photo
        output_path="photo_pattern_qr.png",
        qr_size=1000
    )


✅ Photo-pattern QR saved at photo_pattern_qr.png


In [6]:
import qrcode
from PIL import Image, ImageEnhance
import numpy as np

def generate_photo_qr(data, photo_path, output_path="photo_qr_strong.png",
                      qr_size=800, contrast_boost=1.6, brightness_boost=0.9):
    """
    Generate a QR code using a photo as module fill with auto-enhancement.
    """

    # Step 1: Create QR matrix
    qr = qrcode.QRCode(
        error_correction=qrcode.constants.ERROR_CORRECT_H,
        border=2
    )
    qr.add_data(data)
    qr.make(fit=True)
    matrix = qr.get_matrix()
    n_rows, n_cols = len(matrix), len(matrix[0])

    # Step 2: Load and resize photo
    photo = Image.open(photo_path).convert("RGB")
    photo = photo.resize((n_cols, n_rows), Image.LANCZOS)

    # Step 3: Enhance photo for stronger visibility
    enhancer = ImageEnhance.Contrast(photo)
    photo = enhancer.enhance(contrast_boost)
    enhancer = ImageEnhance.Brightness(photo)
    photo = enhancer.enhance(brightness_boost)

    # Step 4: Fill QR modules with photo pixels
    out_img = Image.new("RGB", (n_cols, n_rows), "white")
    photo_pixels = np.array(photo)

    for y in range(n_rows):
        for x in range(n_cols):
            if matrix[y][x]:  # dark module
                out_img.putpixel((x, y), tuple(photo_pixels[y, x]))
            else:
                out_img.putpixel((x, y), (255, 255, 255))  # white background

    # Step 5: Scale up for clarity
    out_img = out_img.resize((qr_size, qr_size), Image.NEAREST)
    out_img.save(output_path)
    print(f"✅ Photo QR saved at {output_path}")

if __name__ == "__main__":
    generate_photo_qr(
        data="https://sakthijai0005-lang.github.io/revathi/",  # 🔗 replace with your link
        photo_path="SM247500.JPG",             # 📸 your photo
        output_path="photo_qr_strong.png",
        qr_size=1000
    )


✅ Photo QR saved at photo_qr_strong.png


In [8]:
import qrcode
from PIL import Image
import numpy as np

def generate_photo_colored_qr(data, photo_path, output_path="photo_colored_qr.png", qr_size=800):
    """
    Generate a QR code where dark modules are filled with photo colors instead of black.
    """
    # Step 1: Make QR code matrix
    qr = qrcode.QRCode(
        version=None,
        error_correction=qrcode.constants.ERROR_CORRECT_H,
        border=2
    )
    qr.add_data(data)
    qr.make(fit=True)
    matrix = qr.get_matrix()
    n_rows, n_cols = len(matrix), len(matrix[0])

    # Step 2: Load and resize photo to match QR grid
    photo = Image.open(photo_path).convert("RGB")
    photo = photo.resize((n_cols, n_rows), Image.LANCZOS)
    photo_pixels = np.array(photo)

    # Step 3: Create output image
    out_img = Image.new("RGB", (n_cols, n_rows), "white")

    for y in range(n_rows):
        for x in range(n_cols):
            if matrix[y][x]:  # dark square → use photo pixel color
                out_img.putpixel((x, y), tuple(photo_pixels[y, x]))
            else:             # light square → keep white
                out_img.putpixel((x, y), (255, 255, 255))

    # Step 4: Scale up for clarity
    out_img = out_img.resize((qr_size, qr_size), Image.NEAREST)
    out_img.save(output_path)
    print(f"✅ Photo-colored QR saved at {output_path}")

if __name__ == "__main__":
    generate_photo_colored_qr(
        data="https://sakthijai0005-lang.github.io/revathi/",  # 🔗 replace with your link
        photo_path="r2.png",           # 📸 replace with your photo
        output_path="photo_colored_qr.png",
        qr_size=1000
    )


✅ Photo-colored QR saved at photo_colored_qr.png
